# UNO Game AI - Assignment 2
### AI2002 Artificial Intelligence | Ms. Umarah Qaseem

| Player | Algorithm | Strategy |
|--------|-----------|----------|
| P1 | Minimax | Defensive |
| P2 | Expectimax | Offensive |
| P3 | Minimax / Manual | Simulation or User |

**GitHub Repository:** https://github.com/hasnain-afkar/UNO-GameAI  

In [1]:
import random
import copy

## 1. Card Class

In [2]:
# Represents one UNO card
# Color: Red, Blue, Green, Yellow
# Value: 0-9 or Skip
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    # Print card nicely e.g. Red 5
    def __repr__(self):
        return str(self.color) + ' ' + str(self.value)

    # Compare two cards for equality
    def __eq__(self, other):
        return self.color == other.color and self.value == other.value

    # Allow cards to be used in sets and dicts
    def __hash__(self):
        return hash((self.color, self.value))


## 2. Deck Generator

In [3]:
# Creates a full simplified UNO deck and shuffles it
# Each color: numbers 0-9 (two copies) + 2 Skip cards
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []
    # Two copies of each number card per color
    for color in colors:
        for num in range(10):
            deck.append(Card(color, num))
            deck.append(Card(color, num))
    # Two Skip cards per color
    for color in colors:
        deck.append(Card(color, 'Skip'))
        deck.append(Card(color, 'Skip'))
    random.shuffle(deck)
    return deck

deck = generate_deck()
print('Deck size:', len(deck))
print('Sample cards:', deck[:5])


Deck size: 88
Sample cards: [Red 1, Blue 7, Red 3, Blue 9, Green 5]


## 3. Legal Move Generator

In [4]:
# Returns cards from hand that can legally be played on top_card
# Rule: card must match top card color OR value
def get_valid_moves(hand, top_card):
    valid = []
    for card in hand:
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid


## 4. State Transition and Game Setup

### State Representation
The game state dictionary as required by the assignment:

```python
state = {
    'p1'        : ai_cards,          # Player 1 hand (Minimax)
    'p2'        : opponent1_cards,   # Player 2 hand (Expectimax)
    'p3'        : opponent2_cards,   # Player 3 hand (Manual/Sim)
    'top_card'  : top_card,
    'deck'      : deck,
    'skip_next' : False
}
```

In [5]:
# Applies a move and returns the updated state (deep copy)
# card=None means the player draws one card from the deck
def apply_move(state, player_key, card):
    new_state = copy.deepcopy(state)
    if card is None:
        # Draw rule: take one card from top of deck
        if new_state['deck']:
            drawn = new_state['deck'].pop(0)
            new_state[player_key].append(drawn)
        new_state['skip_next'] = False
    else:
        # Play rule: remove from hand and place on pile
        new_state[player_key].remove(card)
        new_state['top_card'] = card
        # Skip rule: set flag if Skip card was played
        new_state['skip_next'] = (card.value == 'Skip')
    return new_state


# Sets up a brand new game state
def setup_game():
    deck = generate_deck()
    # Deal 5 cards each as required by assignment
    p1 = [deck.pop() for _ in range(5)]
    p2 = [deck.pop() for _ in range(5)]
    p3 = [deck.pop() for _ in range(5)]
    # Starting top card must not be a Skip
    top = deck.pop()
    while top.value == 'Skip':
        deck.insert(0, top)
        top = deck.pop()
    return {
        'p1': p1,         # ai_cards
        'p2': p2,         # opponent1_cards
        'p3': p3,         # opponent2_cards
        'top_card': top,
        'deck': deck,
        'skip_next': False
    }

state = setup_game()
print('Top card:', state['top_card'])
print('P1 hand:', state['p1'])
print('P2 hand:', state['p2'])
print('P3 hand:', state['p3'])
print('Valid moves for P1:', get_valid_moves(state['p1'], state['top_card']))


Top card: Blue 8
P1 hand: [Yellow 4, Green 3, Red 9, Yellow 8, Green Skip]
P2 hand: [Yellow 7, Green 2, Green 0, Blue 9, Green 6]
P3 hand: [Blue 2, Red 3, Red 2, Yellow 7, Red Skip]
Valid moves for P1: [Yellow 8]
